<a href="https://colab.research.google.com/github/AbdulWasay65/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbdulWasay65/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes
### Rule reasoning

My baseline rule is designed to prioritize content-refresh opportunities using current-period Google Search Console signals only.

I will check two signals before encoding the rule:

* **Search impressions:** higher impressions indicate greater observed search visibility and a larger potential opportunity for a page to benefit from improvement.
* **Average position:** weaker average position indicates more room for search-position improvement. This signal is related to FlyRank's CTR-vs-position flag logic.

The rule will combine these signals into a simple, transparent priority score. It will use only information available at the decision moment and will not use future performance, labels, or product-flag outputs.

### Reason codes

The rule can assign one primary reason code:

* `HIGH_IMPRESSIONS_WEAK_POSITION` — the page has meaningful search visibility but a relatively weak average position, making it a strong refresh candidate.
* `HIGH_IMPRESSIONS` — the page has substantial search visibility but does not meet the weaker-position condition.
* `WEAK_POSITION` — the page has a relatively weak average position but lower observed search volume.
* `LOW_PRIORITY` — the page does not meet either strong-opportunity condition.

The score is intended for directional decision support and prioritization, not as a causal prediction of future performance.


### Signal check 1 — Search impressions

**Signal:** `gsc_impressions`

**Why I am checking it:** Search impressions represent observed search visibility. I expect pages with higher impression volume to represent larger observable search opportunities for content refresh prioritization.

**Verdict:** CONFIRMED if the bucket distribution shows meaningful variation in impressions and the higher-volume buckets contain a substantial number of rows. The result will be treated as directional rather than causal.


In [ ]:
# This cell is for CODE
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

rel = "hf://datasets/FlyRank/internship-warehouse"

print("Warehouse connection configured.")
print("Development month: 2026-03")


Warehouse connection configured.
Development month: 2026-03


In [ ]:
# Signal check 1 — Search impressions

impressions_query = f"""
SELECT
    CASE
        WHEN gsc_impressions < 10 THEN '<10'
        WHEN gsc_impressions < 100 THEN '10-99'
        WHEN gsc_impressions < 1000 THEN '100-999'
        WHEN gsc_impressions < 10000 THEN '1,000-9,999'
        ELSE '10,000+'
    END AS impressions_bucket,
    COUNT(*) AS n
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
GROUP BY impressions_bucket
ORDER BY
    CASE impressions_bucket
        WHEN '<10' THEN 1
        WHEN '10-99' THEN 2
        WHEN '100-999' THEN 3
        WHEN '1,000-9,999' THEN 4
        WHEN '10,000+' THEN 5
    END
"""

impressions_check = con.sql(impressions_query).df()

display(impressions_check)

print("Signal checked: gsc_impressions")
print("Total rows checked:", impressions_check["n"].sum())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**Verdict: CONFIRMED**

The `gsc_impressions` signal shows substantial variation across the March 2026 GSC-available rows. Most observations are below 1,000 impressions, while a smaller group has much higher observed search visibility. This supports using impressions as a directional opportunity signal in the baseline prioritization rule.


In [ ]:
# Signal check 2 — Average search position

position_query = f"""
SELECT
    CASE
        WHEN gsc_avg_position < 3 THEN '<3'
        WHEN gsc_avg_position < 5 THEN '3-4.99'
        WHEN gsc_avg_position < 10 THEN '5-9.99'
        WHEN gsc_avg_position < 20 THEN '10-19.99'
        ELSE '20+'
    END AS position_bucket,
    COUNT(*) AS n
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
  AND gsc_avg_position IS NOT NULL
GROUP BY position_bucket
ORDER BY
    CASE position_bucket
        WHEN '<3' THEN 1
        WHEN '3-4.99' THEN 2
        WHEN '5-9.99' THEN 3
        WHEN '10-19.99' THEN 4
        WHEN '20+' THEN 5
    END
"""

position_check = con.sql(position_query).df()

display(position_check)

print("Signal checked: gsc_avg_position")
print("Total rows checked:", position_check["n"].sum())

**Verdict: CONFIRMED**

The `gsc_avg_position` signal shows substantial variation across the March 2026 GSC-available rows, ranging from positions below 3 to 20+. A large number of observations are in weaker-position buckets, so this signal provides a useful directional indicator of potential search-position opportunity for content refresh prioritization.

This signal is also linked to FlyRank's CTR-vs-position flag logic, making it relevant to the baseline rule.


### Baseline rule

I will rank content-refresh opportunities using a simple score based only on current March 2026 search-performance signals.

The score gives higher priority to content with:

1. **Higher search impressions**, because more observed search visibility represents a larger potential opportunity.
2. **Weaker average search position**, because these pages have more observed room for improvement.

The rule will assign one primary reason code based on the strongest condition and an action label of `REFRESH`, `MONITOR`, or `LOW_PRIORITY`.

No future-period performance, outcome labels, or existing product flags will be used as inputs.

The score is a transparent baseline for directional decision support. It is not a causal model and does not guarantee that refreshing a page will improve performance.


## 2. Build the ranked queue (writes the CSV)
### Baseline scoring logic

For each content item, I calculate a transparent priority score from current-period GSC signals.

The score combines two components:

* **Impressions component:** higher observed impressions increase the priority score because the page has greater search visibility.
* **Position component:** weaker average position increases the priority score because the page has more observed room for improvement.

The score is used only to rank the available March 2026 content items. It is a baseline decision-support rule, not a prediction of future performance.

Each row receives one primary reason code and one action label:

* `REFRESH` — strong observed opportunity based on visibility and position.
* `MONITOR` — some opportunity is present, but the evidence is weaker.
* `LOW_PRIORITY` — neither signal indicates a strong refresh opportunity.

No future-period fields, labels, or existing FlyRank product flags are used.


In [ ]:
# This cell is for CODE
# Section 2 — Build the ranked baseline action queue

baseline_query = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        AVG(gsc_avg_position) AS avg_position,
        BOOL_OR(gsc_data_available) AS gsc_available
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
),

scored AS (
    SELECT
        client_hash_id,
        content_hash_id,
        impressions,
        clicks,
        avg_position,
        gsc_available,

        -- Higher impressions = higher opportunity.
        -- We use log1p so very large impression values do not dominate.
        LN(1 + impressions) AS impressions_component,

        -- Higher average position number = weaker position = more opportunity.
        LEAST(avg_position, 20) AS position_component

    FROM march
    WHERE avg_position IS NOT NULL
),

final_scores AS (
    SELECT
        *,

        impressions_component + position_component AS priority_score,

        CASE
            WHEN impressions_component >= 5
                 AND position_component >= 10
                THEN 'HIGH_IMPRESSIONS_WEAK_POSITION'

            WHEN impressions_component >= 5
                THEN 'HIGH_IMPRESSIONS'

            WHEN position_component >= 10
                THEN 'WEAK_POSITION'

            ELSE 'LOW_PRIORITY'
        END AS reason_code

    FROM scored
)

SELECT
    ROW_NUMBER() OVER (
        ORDER BY priority_score DESC,
                 client_hash_id,
                 content_hash_id
    ) AS rank,

    client_hash_id,
    content_hash_id,
    impressions,
    clicks,
    avg_position,
    gsc_available,
    priority_score,
    reason_code,

    CASE
        WHEN reason_code = 'HIGH_IMPRESSIONS_WEAK_POSITION'
            THEN 'REFRESH'
        WHEN reason_code IN ('HIGH_IMPRESSIONS', 'WEAK_POSITION')
            THEN 'MONITOR'
        ELSE 'LOW_PRIORITY'
    END AS action

FROM final_scores
ORDER BY rank
"""

baseline_df = con.sql(baseline_query).df()

print("Rows in ranked queue:", len(baseline_df))
display(baseline_df.head(20))
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

### Top-20 review

The ranked queue prioritizes pages with high observed Google Search Console impressions and weaker average search position. All top-20 rows receive the `REFRESH` action with the reason code `HIGH_IMPRESSIONS_WEAK_POSITION`.

| Rank | Action  | Reason code                    | Confidence note                                                                                        | What would make it wrong                                                                                               |
| ---- | ------- | ------------------------------ | ------------------------------------------------------------------------------------------------------ | ---------------------------------------------------------------------------------------------------------------------- |
| 1    | REFRESH | HIGH_IMPRESSIONS_WEAK_POSITION | High impressions with position 32.77 indicate substantial observed search visibility but weak ranking. | The impressions may come from a broad or low-value query mix, so refreshing the page may not improve outcomes.         |
| 2    | REFRESH | HIGH_IMPRESSIONS_WEAK_POSITION | Very high impressions and position 22.56 make this a strong baseline priority.                         | The page may already be appropriate for its search intent despite the weak average position.                           |
| 3    | REFRESH | HIGH_IMPRESSIONS_WEAK_POSITION | High impressions combined with position 23.34 create a clear refresh opportunity under this rule.      | The observed position may reflect many different queries with different intents.                                       |
| 4    | REFRESH | HIGH_IMPRESSIONS_WEAK_POSITION | High search visibility with position 24.36 gives the rule a strong reason to prioritize it.            | A content change might not address the underlying ranking limitation.                                                  |
| 5    | REFRESH | HIGH_IMPRESSIONS_WEAK_POSITION | High impressions and position 30.77 make this one of the strongest rule-based refresh candidates.      | The impressions could be mostly low-intent or difficult-to-convert searches.                                           |
| 6    | REFRESH | HIGH_IMPRESSIONS_WEAK_POSITION | High impressions with position 23.89 indicate meaningful visibility but weak ranking.                  | The page may require technical or authority improvements rather than a content refresh.                                |
| 7    | REFRESH | HIGH_IMPRESSIONS_WEAK_POSITION | Strong impression volume and position 23.73 support the baseline priority.                             | Search demand or ranking conditions could change independently of content quality.                                     |
| 8    | REFRESH | HIGH_IMPRESSIONS_WEAK_POSITION | High impressions with a very weak position of 36.71 make this a clear rule match.                      | Very weak ranking could indicate that the page is not a good match for the queries generating impressions.             |
| 9    | REFRESH | HIGH_IMPRESSIONS_WEAK_POSITION | High impressions and position 22.47 make this a strong observed priority.                              | The page could be ranking for queries where a refresh would have little effect.                                        |
| 10   | REFRESH | HIGH_IMPRESSIONS_WEAK_POSITION | High impressions with position 27.36 indicate substantial visibility but weak ranking.                 | The ranking problem may be caused by competition or site-level factors rather than stale content.                      |
| 11   | REFRESH | HIGH_IMPRESSIONS_WEAK_POSITION | High impressions and position 20.84 fit the rule's intended refresh pattern.                           | The page may already have suitable content and need another type of intervention.                                      |
| 12   | REFRESH | HIGH_IMPRESSIONS_WEAK_POSITION | High impressions with position 24.08 make it a strong rule-based candidate.                            | The low click volume may indicate that the underlying search demand is not valuable enough to justify a refresh.       |
| 13   | REFRESH | HIGH_IMPRESSIONS_WEAK_POSITION | High impressions and position 28.81 indicate weak observed ranking despite substantial visibility.     | The page may be receiving impressions from queries outside its intended topic.                                         |
| 14   | REFRESH | HIGH_IMPRESSIONS_WEAK_POSITION | High impressions with position 29.05 make this a strong baseline priority.                             | A refresh may not solve a ranking issue caused by authority, competition, or technical factors.                        |
| 15   | REFRESH | HIGH_IMPRESSIONS_WEAK_POSITION | High impressions and position 22.43 provide a clear rule-based reason for prioritization.              | The page's current content may already satisfy users even if average position is weak.                                 |
| 16   | REFRESH | HIGH_IMPRESSIONS_WEAK_POSITION | High impressions combined with position 34.64 indicate weak observed ranking.                          | The page may have insufficient relevance or authority that content editing alone cannot fix.                           |
| 17   | REFRESH | HIGH_IMPRESSIONS_WEAK_POSITION | High impressions with position 45.70 make this an extreme match for the rule.                          | The very weak position may indicate that the page is fundamentally mismatched with the queries generating impressions. |
| 18   | REFRESH | HIGH_IMPRESSIONS_WEAK_POSITION | High impressions and position 36.54 make this a strong rule-based refresh candidate.                   | The observed impressions may not represent realistic opportunities for improvement.                                    |
| 19   | REFRESH | HIGH_IMPRESSIONS_WEAK_POSITION | High impressions with position 25.17 support prioritization under the baseline rule.                   | A content refresh may not be the correct intervention for the observed ranking problem.                                |
| 20   | REFRESH | HIGH_IMPRESSIONS_WEAK_POSITION | High impressions and position 43.17 make this a clear match for the rule.                              | The page may be ranking poorly because of factors outside the content itself.                                          |

### Overall review

The top-20 queue is internally consistent with the baseline rule: every selected page has high observed impressions and weak average position. However, these are **rule-based priorities, not guaranteed refresh wins**. The main uncertainty is whether weak ranking is actually addressable through content changes rather than technical, authority, search-intent, or competitive factors.


In [ ]:
# This cell is for CODE
# Section 3 — Display the top-20 ranked queue for review

top20 = baseline_df.head(20).copy()

display(
    top20[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "impressions",
            "clicks",
            "avg_position",
            "gsc_available",
            "priority_score",
            "reason_code",
            "action",
        ]
    ]
)

print("Top-20 rows reviewed:", len(top20))
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

### Weak picks + leakage check

Some top-ranked pages may be weak picks despite scoring highly. A high impression count and weak average position do not prove that a content refresh will improve performance.

The strongest potential weak picks are pages with very high impressions but extremely weak average position. For example, a page ranking around position 36–46 may have substantial impressions but may be receiving those impressions from queries where the page is not a strong relevance match. In such cases, refreshing the content may not solve the underlying problem.

Another weak-pick risk is low click volume despite high impressions. This can indicate that the observed search visibility does not translate into meaningful clicks, so a refresh should not automatically be treated as a guaranteed opportunity.

### Leakage check

The baseline score uses only March 2026 information:

* `gsc_impressions`
* `gsc_clicks`
* `gsc_avg_position`
* `gsc_data_available`

No future-month performance, outcome label, product flag, or label-derived field is used by the ranking rule.

The final queue is therefore a **directional decision-support baseline**, not a prediction of guaranteed future improvement.


In [ ]:
# This cell is for CODE
# Section 4 — Weak picks + leakage check

# Confirm the baseline queue contains only decision-time fields.
baseline_features = [
    "impressions",
    "clicks",
    "avg_position",
    "gsc_available"
]

future_or_label_fields = [
    "label",
    "april_impressions",
    "future_impressions",
    "future_clicks",
    "future_position"
]

present_baseline_features = [
    col for col in baseline_features
    if col in baseline_df.columns
]

present_future_or_label_fields = [
    col for col in future_or_label_fields
    if col in baseline_df.columns
]

print("Decision-time features present:")
print(present_baseline_features)

print("\nFuture/label-derived fields present:")
print(present_future_or_label_fields)

print("\nLeakage check:",
      "PASS" if len(present_future_or_label_fields) == 0 else "FAIL")

# Show several potential weak picks for manual review:
# high impressions but very weak average position.
weak_picks = (
    baseline_df
    .sort_values(
        ["avg_position", "impressions"],
        ascending=[False, False]
    )
    .head(5)
)

print("\nPotential weak picks for review:")
display(
    weak_picks[
        [
            "rank",
            "impressions",
            "clicks",
            "avg_position",
            "priority_score",
            "reason_code",
            "action"
        ]
    ]
)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## Self-check

* [x] Every section is filled with both Markdown reasoning and supporting code.
* [x] The notebook runs successfully through the completed sections without errors.
* [x] No client names, URLs, or private search queries are included.
* [x] Claims are described as observed, measured, directional, and decision-support rather than causal.
* [x] Two signals were checked with visible bucket tables and row counts.
* [x] At least one checked signal is linked to a real FlyRank-style flag: impressions/volume and search position are used in the baseline prioritization rule.
* [x] One baseline rule was encoded with a priority score, reason code, and action label.
* [x] The ranked queue contains 176,738 rows and is generated by the notebook.
* [x] The top 20 ranked rows were reviewed with action, reason, confidence, and possible failure conditions.
* [x] Weak picks were reviewed and the baseline was checked for future or label-derived leakage.
* [x] Leakage check passed: no future or label-derived fields are present in the final baseline queue.
* [x] No future-window information is used by the baseline ranking rule.
* [x] The CSV is generated by the notebook at `work/outputs/baseline_action_score.csv` and is not required to be committed.
